# Stage 1. Frame Selection and Quality Control

Video palm merekam partisipan membuka kepalan tangan secara bertahap sehingga tidak semua frame menampilkan telapak tangan dalam kondisi terbuka dan tajam. Notebook ini menjalankan MediaPipe Hands per video untuk memilih satu frame representatif terbaik lewat src.sites.palm.frame_selection, lalu melaporkan tingkat keberhasilan deteksi dan distribusi skor kualitas, analog dengan gerbang quality control pada stage 1 konjungtiva.

## Environment Setup

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent.parent))

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from configs import paths
from src.sites.palm import frame_selection

output_dir = paths.outputs_dir("palm")
manifest = pd.read_csv(output_dir / "manifest.csv")
frames_dir = output_dir / "frames"
frames_dir.mkdir(parents=True, exist_ok=True)
print("total video", len(manifest))

## Run Frame Selection

Setiap video diproses satu kali. Frame terbaik dan landmark tangannya disimpan ke disk agar tahap segmentasi ROI berikutnya tidak perlu menjalankan ulang MediaPipe Hands.

In [ ]:
records = []
for _, row in manifest.iterrows():
    result = frame_selection.extract_best_frame(row["raw_path"])
    if result is None:
        records.append({"uid": row["uid"], "detected": False, "method": "failed"})
        continue
    frame_path = frames_dir / f"{row['uid']}.png"
    cv2.imwrite(str(frame_path), cv2.cvtColor(result["frame"], cv2.COLOR_RGB2BGR))
    record = {
        "uid": row["uid"],
        "detected": True,
        "method": result["detection_method"],
        "frame_path": str(frame_path),
        "landmarks_path": "",
        "frame_index": result["frame_index"],
        "openness": result["openness"],
        "sharpness": result["sharpness"],
        "brightness": result["brightness"],
    }
    if result["landmarks_px"] is not None:
        landmarks_path = frames_dir / f"{row['uid']}_landmarks.npy"
        np.save(landmarks_path, result["landmarks_px"])
        record["landmarks_path"] = str(landmarks_path)
    records.append(record)

frame_qc = pd.DataFrame(records)
print("detection rate", round(100 * frame_qc["detected"].mean(), 1), "percent")
print(frame_qc["method"].value_counts())

## Undetected Videos

Video yang benar-benar gagal diproses (misalnya file rusak atau tidak terbaca) dilaporkan agar transparan. Video close-up ekstrem yang gagal deteksi landmark MediaPipe tetap dipakai lewat fallback segmentasi warna kulit (`method="skin_fallback"`) sehingga tidak membuat distribusi label bias.

In [ ]:
undetected = frame_qc.loc[~frame_qc["detected"], "uid"].tolist()
print(f"video gagal diproses total: {len(undetected)}")
print(undetected[:10])

## Quality Score Distributions

Distribusi openness, sharpness, dan brightness pada frame terpilih membantu memverifikasi bahwa frame yang dipilih memang representatif (tangan cukup terbuka, tajam, dan pencahayaan wajar).

In [ ]:
detected = frame_qc[frame_qc["detected"]]
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
for ax, column, title in zip(
    axes,
    ["openness", "sharpness", "brightness"],
    ["Hand Openness", "Sharpness (Laplacian variance)", "Brightness"],
):
    ax.hist(detected[column], bins=30, color="seagreen", alpha=0.8)
    ax.set_title(title)
plt.tight_layout()
plt.show()

## Preview Selected Frames

Beberapa contoh frame terpilih beserta landmark tangan untuk verifikasi visual bahwa MediaPipe Hands mendeteksi telapak tangan dengan benar.

In [ ]:
sample_rows = detected.sample(min(4, len(detected)), random_state=42)
fig, axes = plt.subplots(1, len(sample_rows), figsize=(4 * len(sample_rows), 4))
axes = np.atleast_1d(axes)
for ax, (_, row) in zip(axes, sample_rows.iterrows()):
    frame = cv2.cvtColor(cv2.imread(row["frame_path"]), cv2.COLOR_BGR2RGB)
    ax.imshow(frame)
    if row["landmarks_path"]:
        landmarks = np.load(row["landmarks_path"])
        ax.scatter(landmarks[:, 0], landmarks[:, 1], s=10, c="red")
    ax.set_title(f"{row['uid']} ({row['method']})")
    ax.axis("off")
plt.tight_layout()
plt.show()

## Save Frame QC Report

In [ ]:
frame_qc_path = output_dir / "frame_qc.csv"
frame_qc.to_csv(frame_qc_path, index=False)
print("frame qc saved to", frame_qc_path)